# Tokenizer

> Split SMILES with a chemistry-aware regex, then map pieces through IBM's curated BERT vocab.


In [ ]:
#| default_exp tokenizer


In [ ]:
#| hide
from nbdev.showdoc import *


## The idea

English tokenizers split on spaces and subwords. SMILES has no spaces — `CCO` is three atoms glued together. IBM's trick is a **regex** that knows chemical atoms and bonds:

```text
CCO  →  ['C', 'C', 'O']
c1ccccc1  →  ['c', '1', 'c', 'c', 'c', 'c', 'c', '1']
```

Those pieces go through a slow `BertTokenizerLegacy` loaded from `bert_vocab_curated.txt`. Special tokens are remapped to the SMI-TED names: `<bos>`, `<eos>`, `<pad>`, `<mask>`.


In [ ]:
#| export
from __future__ import annotations

import os
import re
from typing import TYPE_CHECKING

from huggingface_hub import hf_hub_download
from max.pipelines.lib import TextTokenizer
from transformers.models.bert.tokenization_bert_legacy import BertTokenizerLegacy

if TYPE_CHECKING:
    from max.pipelines.lib.config import PipelineConfig


## The SMILES regex

This pattern is the heart of the tokenizer. Brackets first (`[Na+]`), then two-letter halogens, then single letters, then punctuation and ring digits.


In [ ]:
#| export
_SMILES_PATTERN = re.compile(
    r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])"
)


In [ ]:
#| hide
assert _SMILES_PATTERN.findall("CCO") == ["C", "C", "O"]
assert _SMILES_PATTERN.findall("c1ccccc1")[0] == "c"
_SMILES_PATTERN.findall("CCO")


## MolTranBertTokenizer

Subclass the **slow** `BertTokenizerLegacy` (transformers ≥5 made `BertTokenizer` a fast WordPiece backend that ignores `_tokenize`). Override `_tokenize` so WordPiece never sees raw SMILES — only the regex pieces. Null out `wordpiece_tokenizer` / `basic_tokenizer` like the IBM reference.


In [ ]:
#| export
class MolTranBertTokenizer(BertTokenizerLegacy):
    "Regex SMILES tokenizer with the curated SMI-TED vocabulary."

    def __init__(self, vocab_file: str = "", **kwargs) -> None:
        super().__init__(
            vocab_file,
            unk_token="<pad>",
            sep_token="<eos>",
            pad_token="<pad>",
            cls_token="<bos>",
            mask_token="<mask>",
            do_lower_case=False,
            **kwargs,
        )
        # Match IBM: disable the slow BERT WordPiece / basic pipeline.
        self.wordpiece_tokenizer = None
        self.basic_tokenizer = None

    def _tokenize(self, text: str) -> list[str]:
        return _SMILES_PATTERN.findall(text)


## SmiTedTokenizer — the MAX wrapper

MAX expects a `TextTokenizer`. We load the vocab from the model directory (or download it from the Hub), wrap `MolTranBertTokenizer`, and expose `eos` for the pipeline.


In [ ]:
#| export
class SmiTedTokenizer(TextTokenizer):
    "MAX ``TextTokenizer`` that delegates to ``MolTranBertTokenizer``."

    def __init__(
        self,
        model_path: str,
        pipeline_config: PipelineConfig,
        *,
        revision: str | None = None,
        max_length: int | None = None,
        trust_remote_code: bool = False,
        enable_llama_whitespace_fix: bool = False,
        chat_template: str | None = None,
        **unused_kwargs,
    ) -> None:
        self.model_path = model_path
        vocab_file = os.path.join(model_path, "bert_vocab_curated.txt")
        if not os.path.isfile(vocab_file):
            vocab_file = hf_hub_download(
                repo_id=model_path,
                filename="bert_vocab_curated.txt",
                revision=revision,
            )

        self.delegate = MolTranBertTokenizer(
            vocab_file=vocab_file,
            model_max_length=max_length or 202,
        )
        self._custom_template_provided = chat_template is not None
        if chat_template is not None:
            self.delegate.chat_template = chat_template
        self.max_length = max_length or self.delegate.model_max_length
        self._enable_llama_whitespace_fix = False
        self._llama_whitespace_fix_dummy_token_id = 0
        self._llama_whitespace_fix_dummy_token_len = 0
        self._default_eos_token_ids = {self.eos}

    @property
    def eos(self) -> int:
        if self.delegate.sep_token_id is not None:
            return int(self.delegate.sep_token_id)
        return 0


Try it on ethanol if the local model assets are present:


In [ ]:
#| hide
from pathlib import Path
assets = Path("../model_assets/ibm-research_materials.smi-ted/bert_vocab_curated.txt")
if assets.is_file():
    t = MolTranBertTokenizer(vocab_file=str(assets), model_max_length=202)
    print(t.tokenize("CCO"))
    print(t.encode("CCO", add_special_tokens=True)[:12])
else:
    print("model assets missing; skip live tokenize demo")


### Checkpoint

*SMILES has no spaces — a chemistry regex makes the tokens, a curated vocab makes the ids.*


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
